# Phase 6b — device-resident COCO128 training + GPU speed
Full yolov8n training loop (dataset -> minibatch -> device fwd/bwd/Adam + host v8 loss)
over COCO128, measuring **seconds/epoch on GPU** vs the hosted-GEMM baseline (~4.6 min/epoch).
**Runtime → GPU (T4)**, Run all.


In [ ]:
!nvidia-smi -L


In [ ]:
%cd /content
!rm -rf yolov8_cpp
!git clone -q --branch thrust-backend https://github.com/yomei-o/yolov8_cpp.git
%cd /content/yolov8_cpp


### Get COCO128


In [ ]:
!wget -q https://github.com/ultralytics/yolov5/releases/download/v1.0/coco128.zip
!unzip -q -o coco128.zip -d /content/yolov8_cpp
!ls coco128/images/train2017 | wc -l


### Build (GPU) + train (imgsz 640, batch 4, 3 epochs) — watch the s/epoch


In [ ]:
!nvcc -x cu -O2 -std=c++17 --extended-lambda -arch=native -DUSE_CUDA -Ipure/third_party pure/dtrain_coco.cpp -o dtrain_coco
!./dtrain_coco coco128/images/train2017 640 4 3


---
**What to look for**: the `s/epoch` number. Compare to the hosted-GEMM baseline **~4.6 min/epoch**
(COCO128/640/batch4). If host RAM/GPU OOMs, lower to `... 640 2 3` or `... 320 4 3`.
The loss should also decrease across the 3 epochs (real v8 loss).
